# 02 — Pré-processamento e Análise de Correlação

**MedAI Triage** · Fase 3 do desafio

Este notebook transforma os dados brutos em conjuntos prontos para modelagem e investiga a estrutura de dependência entre as features.

**A questão central da fase é a integridade do experimento:** toda estatística aprendida dos dados — mediana da imputação, média e desvio do escalonamento — precisa vir **exclusivamente do conjunto de treino**. Com 569 amostras, vazar informação do teste é fácil de fazer e quase impossível de perceber depois: as métricas simplesmente ficam otimistas, sem nenhum sinal de erro.

| Seção | Pergunta |
|---|---|
| 1 | O que a limpeza precisa fazer, e o que ela **não** deve fazer? |
| 2 | Como separar treino, validação e teste sem contaminar nada? |
| 3 | Como montar um pipeline que não vaza e que vai para produção? |
| 4 | Quão redundantes são as features? |
| 5 | Vale construir um conjunto reduzido? |

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

from medai import viz
from medai.config import carregar_config
from medai.data.loader import carregar_dados_brutos
from medai.data.split import construir_cv, particionar, salvar_particoes
from medai.features import correlation as corr
from medai.features.build_pipeline import construir_preprocessador, nomes_features_saida
from medai.features.cleaning import limpar, separar_alvo

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
viz.configurar_estilo(interativo=True)

config = carregar_config(RAIZ / "configs" / "breast_cancer.yaml")
df_bruto = carregar_dados_brutos(config)
print(f"Dataset bruto: {df_bruto.shape}")

---
## 1. Limpeza

A limpeza é deliberadamente **restrita a operações que não aprendem nada dos dados**: descartar colunas, mapear o rótulo, remover duplicatas, converter zeros impossíveis em ausentes.

Tudo que *estima* algo — imputação, escalonamento, codificação — fica no pipeline do scikit-learn, ajustado só no treino. Essa fronteira é o ponto que decide se o experimento é válido.

In [ ]:
df, relatorio = limpar(df_bruto, config)
print(relatorio)

### Leitura

O CSV presente veio da rota `sklearn`, que não possui as colunas `id` nem `Unnamed: 32` — por isso elas aparecem como "descartes não aplicáveis" em vez de "descartadas". **A regra está implementada e testada**; ela simplesmente não encontra o que remover nesta rota. Com o CSV do Kaggle, as duas seriam efetivamente descartadas.

Vale explicitar por que `id` precisa sair: não tem significado clínico e, se os casos tiverem sido coletados em ordem correlacionada com o desfecho (por exemplo, uma campanha de rastreamento concentrada em certo período), o modelo aprenderia o número de registro em vez da patologia. Seria um vazamento invisível nas métricas e inútil em produção.

**Nenhum valor ausente e nenhuma duplicata** neste dataset. A imputação entra no pipeline mesmo assim — ver seção 3.

In [ ]:
X, y = separar_alvo(df, config)
print(f"X: {X.shape}")
print(f"y: {y.value_counts().to_dict()}  (1 = maligno = classe positiva)")
print(f"Taxa de positivos: {y.mean():.1%}")

---
## 2. Particionamento — antes de qualquer ajuste

Três conjuntos, cada um com um papel que não se mistura:

| Conjunto | Papel | Quando é usado |
|---|---|---|
| **Treino** (60%) | Ajustar o modelo; a CV roda aqui dentro | Fase 4 |
| **Validação** (20%) | Escolher o limiar de decisão, conferir a seleção | Fase 4 |
| **Teste** (20%) | Estimar o desempenho | **Uma única vez**, Fase 6 |

Por que três e não dois: se o limiar de decisão fosse escolhido no teste, a estimativa final estaria otimista e **não haveria dado reservado para corrigi-la**. O erro seria permanente.

In [ ]:
dados = particionar(X, y, config)
print(dados)

In [ ]:
# A estratificação preservou a proporção de malignos nos três conjuntos?
comparacao = pd.DataFrame({
    "n": dados.tamanhos,
    "taxa_positivos": dados.taxas_positivas,
})
comparacao["desvio_pp"] = ((comparacao["taxa_positivos"] - y.mean()) * 100).round(2)
comparacao

In [ ]:
# Os conjuntos são realmente disjuntos?
treino, validacao, teste = (set(d.index) for d in (dados.x_treino, dados.x_validacao, dados.x_teste))
print(f"treino ∩ validação: {len(treino & validacao)}")
print(f"treino ∩ teste:     {len(treino & teste)}")
print(f"validação ∩ teste:  {len(validacao & teste)}")
print(f"união:              {len(treino | validacao | teste)} de {len(X)}")

### Leitura

A estratificação manteve a taxa de positivos dentro de **0,5 ponto percentual** em todos os conjuntos. Sem ela, com apenas 114 amostras no teste, uma divisão aleatória infeliz poderia produzir um conjunto com proporção bem diferente — e as métricas refletiriam a sorte do sorteio, não o modelo.

Os três conjuntos são disjuntos e cobrem o dataset inteiro.

**Sobre validação cruzada:** com treino de 341 amostras, usar a CV estratificada de 5 folds *dentro do treino* extrai mais sinal do que confiar em um holdout único. Na CV cada amostra serve de validação em exatamente um fold; num holdout, 20% das amostras validam sempre e 80% nunca. O conjunto de validação separado fica para a escolha do limiar — uma decisão que a CV não resolve bem, porque precisa ser feita uma vez só, sobre um conjunto fixo.

In [ ]:
cv = construir_cv(config)
print(cv)

# Cada fold preserva a proporção de classes?
folds = []
for i, (idx_treino, idx_val) in enumerate(cv.split(dados.x_treino, dados.y_treino), 1):
    folds.append({
        "fold": i,
        "n_treino": len(idx_treino),
        "n_val": len(idx_val),
        "pos_treino": round(dados.y_treino.iloc[idx_treino].mean(), 3),
        "pos_val": round(dados.y_treino.iloc[idx_val].mean(), 3),
    })
pd.DataFrame(folds)

---
## 3. Pipeline de pré-processamento

Um `ColumnTransformer` com um ramo por tipo de variável:

- **numérico:** `SimpleImputer(median)` → `StandardScaler()`
- **categórico:** `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown='ignore')`

Este dataset é inteiramente numérico, mas o ramo categórico existe por duas razões: cumpre o requisito de converter variáveis categóricas e numéricas em formatos adequados, e permite rodar o mesmo pipeline sobre outro dataset sem tocar no código.

In [ ]:
preprocessador = construir_preprocessador(X, config)
preprocessador

In [ ]:
# Ajuste EXCLUSIVAMENTE no treino.
preprocessador.fit(dados.x_treino)

treino_t = preprocessador.transform(dados.x_treino)
teste_t = preprocessador.transform(dados.x_teste)

print("Após o escalonamento:")
print(f"  treino — média: {treino_t.mean():+.2e}   desvio: {treino_t.std():.4f}")
print(f"  teste  — média: {teste_t.mean():+.4f}   desvio: {teste_t.std():.4f}")

### Leitura — a verificação mais importante da fase

O treino sai com média **exatamente zero** e desvio **exatamente 1**: é a definição do `StandardScaler`.

O teste **não** sai centrado em zero, e isso é o resultado correto. Ele foi transformado com a média e o desvio *do treino*, e é uma amostra diferente — portanto desvia um pouco. 

Se o teste também saísse com média zero, significaria que o escalonador tinha visto os dados de teste ao ser ajustado. Nenhuma exceção seria lançada, nenhum aviso apareceria, e as métricas da Fase 6 ficariam otimistas sem que nada indicasse o problema. É o erro mais caro e mais silencioso deste tipo de projeto, e existe um teste automatizado (`test_pipeline_nao_vaza_estatisticas_do_teste`) que falha se alguém o introduzir.

In [ ]:
# Por que a imputação permanece, mesmo sem ausentes neste dataset.
exemplo = dados.x_teste.head(1).copy()
exemplo.iloc[0, 0] = np.nan  # simula um payload incompleto chegando na API

transformado = preprocessador.transform(exemplo)
print(f"Entrada com NaN em '{exemplo.columns[0]}' → saída contém NaN? {np.isnan(transformado).any()}")
print(f"Valor imputado (padronizado): {transformado[0, 0]:+.4f}")
print(f"Mediana do treino usada: {dados.x_treino.iloc[:, 0].median():.3f}")

### Por que imputar um dataset sem ausentes

Três razões, nenhuma delas "porque o requisito pede":

1. **A API receberá payloads incompletos.** Um médico preenchendo o formulário pode não ter todas as 30 medidas. Sem imputador, a requisição falha; com ele, o sistema responde e sinaliza a incerteza.
2. **O mesmo pipeline roda sobre o Pima Diabetes**, onde `Glucose=0` e `BMI=0` são ausentes disfarçados de zero.
3. **Mediana e não média**, porque as features `_se` têm skew acima de 5 — a média seria puxada pelas caudas.

Note que a mediana usada é **a do treino**, calculada no `fit`. Usar a mediana do próprio lote que chega em produção seria outra forma de vazamento, e produziria predições que mudam conforme quem mais está na fila.

In [ ]:
nomes = nomes_features_saida(preprocessador)
print(f"{len(nomes)} features na saída, na ordem que o modelo vê:")
print(nomes[:6], "...")

---
## 4. Análise de correlação

In [ ]:
# Toda a análise usa SOMENTE o treino: ela informará decisões de modelagem.
X_treino = dados.x_treino
y_treino = dados.y_treino

pares = corr.pares_correlacionados(X_treino, limiar=config.correlacao.limiar_redundancia)
print(f"{len(pares)} pares com |r| ≥ {config.correlacao.limiar_redundancia}")
pares.head(14)

In [ ]:
fig = viz.plot_heatmap_correlacao(X_treino, X_treino.columns.tolist(), metodo="pearson",
                                  titulo="Correlação de Pearson — conjunto de treino")

In [ ]:
# Spearman capta associação monotônica e é robusto à assimetria das features _se.
pearson = corr.matriz_correlacao(X_treino, "pearson").abs()
spearman = corr.matriz_correlacao(X_treino, "spearman").abs()

mascara = np.triu(np.ones(pearson.shape), k=1).astype(bool)
divergencia = (spearman - pearson).abs().where(mascara).stack().sort_values(ascending=False)
divergencia.head(8).round(3).to_frame("|Spearman − Pearson|")

### Leitura

Os pares no topo não são descoberta empírica — são **identidade geométrica**. `perimeter ≈ 2πr` e `area ≈ πr²`, então `radius_mean` e `perimeter_mean` correlacionam a **0,998**. O trio se repete nos três sufixos.

A divergência entre Pearson e Spearman é modesta (máximo de 0,22, em pares envolvendo `concavity_se`), o que indica que as relações são predominantemente **lineares**, não apenas monotônicas. Isso é informação útil: significa que a regressão logística tem chance real de capturar a estrutura, e que transformações não lineares teriam pouco a acrescentar.

In [ ]:
associacao = corr.correlacao_com_alvo(X_treino, y_treino)
associacao.head(10).round(4)

### Correlação com o alvo

`concave points_worst` lidera também aqui (r ≈ 0,79), confirmando o que o d de Cohen mostrou na Fase 2 por outro caminho. A convergência de duas medidas independentes sobre a mesma feature é um sinal de que o achado é robusto, não artefato de um estimador específico.

---
## 5. Multicolinearidade (VIF) e conjunto reduzido

A correlação par a par não conta a história toda. O **VIF** mede quanto a variância do coeficiente de uma feature é inflada pela colinearidade com **todas** as outras em conjunto — uma feature pode ter correlação apenas moderada com cada uma individualmente e ainda assim ser quase uma combinação linear do grupo.

Convenção: VIF > 5 é preocupante, VIF > 10 é severo.

In [ ]:
vif = corr.calcular_vif(X_treino)
print(vif["severidade"].value_counts().to_dict())
vif.head(10).round(1)

### Leitura

**`perimeter_mean` tem VIF de 3.904.** A convenção considera 10 severo. Vinte e cinco das trinta features estão na faixa severa.

Repetindo o ponto porque ele é contraintuitivo: **isso não prejudica a acurácia** dos modelos que vamos treinar. Árvores e boosting escolhem uma feature do grupo redundante e ignoram as demais; a regressão logística com regularização também lida razoavelmente. O que a multicolinearidade destrói é a **interpretação**:

- os coeficientes da regressão logística ficam instáveis — pequenas mudanças no treino alteram magnitudes e podem até inverter sinais;
- o SHAP divide o crédito entre as features correlacionadas, de modo que **cada uma parece menos importante do que o grupo realmente é**.

Como o projeto exige interpretabilidade como entregável, isso importa.

In [ ]:
selecao = corr.selecionar_por_redundancia(X_treino, y_treino, config)
print(selecao)

In [ ]:
vif_reduzido = corr.calcular_vif(X_treino[selecao.mantidas])

comparacao = pd.DataFrame({
    "completo (30)": vif["severidade"].value_counts(),
    "reduzido (24)": vif_reduzido["severidade"].value_counts(),
}).fillna(0).astype(int)
print(comparacao.to_string())
print(f"\nVIF máximo: {vif['vif'].max():.0f} → {vif_reduzido['vif'].max():.0f}")

In [ ]:
vif_reduzido.head(8).round(1)

### Leitura — e uma ressalva honesta

A redução de 30 para 24 features derruba o VIF máximo de **3.904 para 68** — uma queda de duas ordens de magnitude. As seis removidas são exatamente as do trio geométrico redundante, e em cada par preservamos a feature mais associada ao alvo.

**Mas o problema não desaparece.** Dezesseis features continuam na faixa severa, agora lideradas por `concavity_mean` (VIF 68) e `concave points_mean` (67,8). A razão é que `concavity`, `compactness` e `concave points` também medem aspectos relacionados do mesmo fenômeno — irregularidade do contorno nuclear —, correlacionando-se na faixa de 0,8 a 0,9. Elas ficam **abaixo do limiar de 0,95** e por isso sobrevivem ao corte.

Baixar o limiar removeria mais features, mas a um custo: a partir de certo ponto estaríamos descartando informação genuinamente complementar. O limiar de 0,95 foi escolhido para capturar **redundância estrutural** (identidade geométrica), não correlação clínica legítima — são coisas diferentes, e tratá-las igual seria perder sinal.

**A decisão, portanto, não é "qual conjunto é melhor" decidido a priori.** Os dois seguem para a Fase 4 e a comparação é empírica (ADR-009): se o conjunto reduzido mantiver o desempenho, ganhamos interpretabilidade de graça; se perder, teremos quantificado o preço da clareza.

In [ ]:
# Persistência para as próximas fases.
arquivos = salvar_particoes(dados, config)
for nome, caminho in arquivos.items():
    print(f"{nome:>10}: {Path(caminho).name}")

destino = config.caminhos.absoluto("metricas")
vif.to_csv(destino / "fase3_vif_completo.csv")
vif_reduzido.to_csv(destino / "fase3_vif_reduzido.csv")
pares.to_csv(destino / "fase3_pares_redundantes.csv", index=False)
associacao.to_csv(destino / "fase3_correlacao_alvo.csv")

import json
(destino / "fase3_feature_sets.json").write_text(json.dumps({
    "completo": X_treino.columns.tolist(),
    "reduzido": selecao.mantidas,
    "removidas": selecao.removidas,
    "motivo_remocao": {k: {"redundante_com": v[0], "r": v[1]} for k, v in selecao.motivo.items()},
    "vif_max_completo": float(vif["vif"].max()),
    "vif_max_reduzido": float(vif_reduzido["vif"].max()),
}, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nFeature sets gravados em reports/metrics/fase3_feature_sets.json")

---
## 6. Síntese

### Decisões tomadas

| Decisão | Justificativa |
|---|---|
| `id` e `Unnamed: 32` descartadas | Identificador vaza; coluna nula não informa |
| Duplicatas removidas (nenhuma encontrada) | Inflariam artificialmente o peso de uma amostra |
| **Outliers mantidos** | 53,8% dos malignos são atípicos (ADR-007) |
| Split 60/20/20 estratificado, `random_state=42` | Taxa de positivos dentro de 0,5 pp nos três conjuntos |
| CV 5-fold **dentro do treino** | Extrai mais sinal de 341 amostras que um holdout |
| Imputação mediana, mesmo sem ausentes | API receberá payloads incompletos; mediana resiste à assimetria |
| `StandardScaler` em todas as numéricas | Escalas diferem em 6 ordens de magnitude |
| `OneHotEncoder(handle_unknown='ignore')` | Categoria nova em produção não derruba a API |
| **Dois feature sets** seguem para a Fase 4 | Comparação empírica, não decisão a priori |

### Números a reter

- VIF máximo: **3.904** (completo) → **68** (reduzido)
- Features em faixa severa: **25/30** → **16/24**
- Divergência máxima Pearson × Spearman: **0,22** — relações predominantemente lineares
- `concave points_worst` lidera a associação com o alvo (r ≈ 0,79), confirmando o d de Cohen da Fase 2

### Hipóteses abertas para a Fase 4

1. O conjunto reduzido perde desempenho, ou só ganha interpretabilidade?
2. Transformação logarítmica nas features assimétricas ajuda os modelos lineares?
3. `class_weight='balanced'` melhora o recall o bastante para compensar a perda de precisão?